In [ ]:
# %%
# 🧱 1️⃣ Install Required Libraries
# ------------------------------------------------------------------
# We only install the essential dependencies for fine-tuning and retrieval.
# - transformers → for model + tokenizer
# - peft → for LoRA fine-tuning (parameter-efficient)
# - accelerate → for managing distributed/hybrid training
# - sentence-transformers → for encoding text to embeddings
# - faiss → for efficient vector similarity search
# - datasets → for easy dataset loading and creation
# ------------------------------------------------------------------
!pip install -U transformers==4.43.3 peft==0.11.1 accelerate==0.34.2 sentence-transformers==3.0.1
!pip install faiss-cpu datasets

In [ ]:
# %%
# 🧩 2️⃣ Import Libraries
# ------------------------------------------------------------------
# Here we import all necessary Python and ML packages:
# - PyTorch, Hugging Face Transformers, Sentence Transformers, FAISS
# - PEFT (for LoRA fine-tuning)
# ------------------------------------------------------------------
import os
import random
import torch
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel


In [ ]:
# %%
# 📘 3️⃣ Build a Corpus and FAISS Retrieval Index
# ------------------------------------------------------------------
# RAFT (Retrieval-Augmented Fine-Tuning) depends on retrieving relevant
# context for each query. We’ll:
#   1. Load a small text corpus (AG News dataset).
#   2. Convert documents into embeddings.
#   3. Build a FAISS index to perform cosine similarity searches.
# ------------------------------------------------------------------

# Load a small subset (for speed)
corpus_dataset = load_dataset("ag_news", split="train[:500]")
corpus_texts = [f"{row['text']}" for row in corpus_dataset]
print(f"Corpus loaded with {len(corpus_texts)} documents.")

# Use MiniLM sentence encoder (small & fast)
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding corpus...")
corpus_embeddings = embedder.encode(corpus_texts, convert_to_numpy=True, show_progress_bar=True)

# Normalize embeddings for cosine similarity
corpus_embeddings = corpus_embeddings / ((corpus_embeddings**2).sum(axis=1, keepdims=True) ** 0.5)

# Create FAISS index (cosine similarity = inner product on normalized vectors)
dimension = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(corpus_embeddings)

print(f"FAISS cosine-similarity index built with {index.ntotal} vectors.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding corpus...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

FAISS cosine-similarity index built with 500 vectors.


In [ ]:
# %%
# 🔍 4️⃣ Retrieval Function
# ------------------------------------------------------------------
# This helper function takes a query string and retrieves top-k similar
# documents from the FAISS index.
# ------------------------------------------------------------------
def retrieve_docs(query, k=3):
    """Retrieve top-k docs using cosine similarity."""
    query_emb = embedder.encode([query], convert_to_numpy=True)
    query_emb = query_emb / ((query_emb**2).sum(axis=1, keepdims=True) ** 0.5)
    distances, indices = index.search(query_emb, k)
    return [corpus_texts[i] for i in indices[0]]


In [ ]:
# %%
# 🧠 5️⃣ Generate Synthetic RAFT Dataset
# ------------------------------------------------------------------
# We create synthetic examples for training:
#   - A question (query)
#   - Retrieved documents (context)
#   - An instruction asking the model to reason and answer
#
# Some examples include a "golden document" containing the correct answer.
# The output is structured with:
#   ##Reason: {reason}
#   ##Answer: {answer}
# ------------------------------------------------------------------

def generate_retrieval_raft_dataset(num_examples=100):
    dataset = []
    qa_pairs = [
        ("Who was the president of the US in 2020?", "Donald Trump"),
        ("What is the capital of Japan?", "Tokyo"),
        ("Who wrote the play Hamlet?", "William Shakespeare"),
        ("Which company created the iPhone?", "Apple"),
        ("What is the largest ocean on Earth?", "Pacific Ocean"),
        ("When did World War II end?", "1945"),
        ("Who discovered gravity?", "Isaac Newton"),
    ]

    instruction = (
        "Instruction: Given the question and retrieved context, provide reasoning and final answer. "
        "Use format:\n##Reason: {reason}\n##Answer: {answer}"
    )

    for _ in tqdm(range(num_examples)):
        q, correct_answer = random.choice(qa_pairs)
        retrieved_docs = retrieve_docs(q, k=5)

        # 80% of samples include the correct document
        use_golden = random.random() < 0.8
        if use_golden:
            golden_doc = f"This document contains the correct answer: {correct_answer}."
            retrieved_docs.append(golden_doc)
            random.shuffle(retrieved_docs)

        context_str = "\n".join(f"[Document {j+1}: {doc}]" for j, doc in enumerate(retrieved_docs))
        input_text = f"Question: {q}\nContext: {context_str}\n{instruction}"
        output_text = (
            f"##Reason: Based on the provided documents, the correct answer is {correct_answer}.\n"
            f"##Answer: {correct_answer}"
        )

        dataset.append({"input": input_text, "output": output_text})

    return Dataset.from_list(dataset)

print("Generating retrieval-augmented RAFT dataset...")
raft_dataset = generate_retrieval_raft_dataset(num_examples=200)
print("✅ Dataset generated successfully!")


Generating retrieval-augmented RAFT dataset...


100%|██████████| 200/200 [00:01<00:00, 102.56it/s]

✅ Dataset generated successfully!


In [ ]:
# %%
# 🤖 6️⃣ Load TinyLlama Model
# ------------------------------------------------------------------
# We use TinyLlama (1.1B parameters) for lightweight fine-tuning.
# LoRA is applied for parameter-efficient training, modifying only
# attention projection layers instead of full model weights.
# ------------------------------------------------------------------

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

# Apply LoRA adapters
model = prepare_model_for_kbit_training(model)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
# %%
# ✂️ 7️⃣ Tokenize Dataset
# ------------------------------------------------------------------
# The dataset is tokenized into model input IDs.
# Each sample concatenates input and output texts,
# followed by an end-of-sequence token.
# ------------------------------------------------------------------

def tokenize_function(examples):
    full_text = [f"{i}{o}{tokenizer.eos_token}" for i, o in zip(examples["input"], examples["output"])]
    tokenized = tokenizer(full_text, truncation=True, max_length=512, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("Tokenizing dataset...")
tokenized_dataset = raft_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])
print("✅ Tokenization complete.")


Tokenizing dataset...


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Tokenization complete.


In [ ]:
# %%
# 🧩 8️⃣ Fine-tune Model with LoRA
# ------------------------------------------------------------------
# We use Hugging Face's Trainer API for simplicity.
# - Only LoRA parameters are updated.
# - Base model stays frozen.
# ------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./raft_faiss_results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("🚀 Starting RAFT fine-tuning...")
trainer.train()
trainer.save_model("./fine_tuned_raft_faiss_adapter")
print("✅ Fine-tuned adapter saved!")


🚀 Starting RAFT fine-tuning...


Step,Training Loss
10,3.300600
20,2.368500
30,2.178100
40,2.013200
50,1.815300


✅ Fine-tuned adapter saved!


In [ ]:
# %%
# 🧪 9️⃣ Inference Test
# ------------------------------------------------------------------
# Test the fine-tuned model:
#   1. Retrieve context for a new query.
#   2. Generate a reasoning + answer output.
# ------------------------------------------------------------------

adapter_path = "./fine_tuned_raft_faiss_adapter"
print("Loading fine-tuned model...")

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
model = PeftModel.from_pretrained(base_model, adapter_path).eval()

query = "Who discovered gravity?"
retrieved_docs = retrieve_docs(query, k=3)
context = "\n".join(f"[Doc {i+1}: {d}]" for i, d in enumerate(retrieved_docs))

instruction = (
    "Instruction: Given the question and retrieved context, provide reasoning and final answer. "
    "Use format: ##Reason: {reason}\n##Answer: {answer}"
)

prompt = f"Question: {query}\nContext:\n{context}\n{instruction}"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=120)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- 🧠 Model Response ---")
print(response)


Loading fine-tuned model...

--- 🧠 Model Response ---
##Reason: According to the retrieved context, the correct answer is Neil Armstrong.
##Answer: Neil Armstrong
